
# Classify the flower type: 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import  matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load the Iris dataset
# iris = load_iris()
# X,y = iris.data, iris.target
# import pandas as pd
# # # Create a DataFrame for better visualization
# # df = pd.DataFrame(iris.data, columns=iris.feature_names)
# # df['species'] = iris.target
# # print(iris.DESCR)

def load_data():
    # Load the Iris dataset
    iris = load_iris()
    features = iris.feature_names
    target = iris.target_names
    X, y = iris.data, iris.target
    return X, y,features, target

X,y,feature_names,target_names = load_data()

In [ ]:
print(feature_names)
print(target_names)
print(f"X.shape:{X.shape} \n yshape:{y.shape}")
print(X[:5]) 
print(y[75:100])

In [ ]:
X_train, x_ , y_train, y_ = train_test_split(X,y, test_size=0.4, random_state=42) 
X_test, X_cv, y_test, y_cv = train_test_split(x_,y_, test_size=0.5, random_state=42)
del x_, y_ # delete theese temporary variables to free up memory #GOOD PRACTICE
# X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42) 
print(f"X_train shape: {X_train.shape} \n y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} \n y_test shape: {y_test.shape}")
print(f"X_cv shape: {X_cv.shape} \n y_cv shape: {y_cv.shape}")

In [ ]:
#  Plot data
def plot_iris_data(X, y, dataset_part_name= None):
    fig, ax = plt.subplots(1,2, figsize=(11.6,4.8))
    # ax = ax.flatten()
    for j in range(2):
        if j == 0:
            xcol = 0
            ax[j].set_title("Sepal Length vs Sepal Width")
        else:
            xcol = 2
            ax[j].set_title("Petal Length vs Petal Width")

        for i in range(3):
            ax[j].scatter(X[y==i,xcol], X[y==i,xcol+1], label=target_names[i])
        ax[j].set_xlabel(feature_names[xcol])
        ax[j].set_ylabel(feature_names[xcol+1])
        ax[j].legend()
    fig.suptitle(f"Iris Dataset Visualization for : {dataset_part_name}")
    # Adjust the layout to prevent overlap from supertitle
    # plt.subplots_adjust(top=0.85, wspace=0.3, hspace=0.1)
    plt.show()  

In [ ]:
plot_iris_data(X_train, y_train, dataset_part_name="Training Set")
plot_iris_data(X_test, y_test, dataset_part_name="Test Set")
plot_iris_data(X_cv, y_cv, dataset_part_name="Cross-Validation Set")


In [ ]:
# normalize the data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # later on after we have trained the model on the training data

### After watching the plot we can say: there is no need of scaling here because features are alreday in comparabale range of ecah other.
- so  scaling does not benefit.
- but another effect of the scaling that:
- after scaling length and width become ``negative (-ve)``, that is not possible to have length in minus. 

### if after normalization getting feature -ve and this may not work for you.
- use **``MinMaxScaler**`` in place of **``StandardScaler**``
- ``from sklearn.preprocessing import MinMaxScaler``
```python
# 1. Create the MinMax Scaler
scaler = MinMaxScaler()

# 2. Scale the data (Notice no negatives will be created!)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Now run your GridSearchCV on the scaled data
grid_search.fit(X_train_scaled, y_train)

```

In [ ]:
plot_iris_data(X_train_scaled, y_train, dataset_part_name="Scaled Training Set")

###  Below cell is just uses the train, test  data.
- The Accuracy results may be misleading: i.e. 100%.
- we should go for: usage of the Croos validation data set also. 
 - X_cv,y_cv alreday splitted.

In [ ]:
lr = LogisticRegression()
# lr.fit(X_train_scaled, y_train)

# fit the model on the unscaled data to see the difference in performance
lr.fit(X_train, y_train)
#  predict on the test data
y_pred = lr.predict(X_test)

# count the number of mismatches between the predicted and actual labels
mismatches = y_test != y_pred
print(f"Number of mismatches: {mismatches.sum()}")

print(classification_report(y_test, y_pred, target_names=target_names))

print(confusion_matrix(y_test, y_pred))

In [ ]:
# x_test1 = [[5.1, 3.5, 1.4, 0.2]]  # example test data

x_test1 = [[1, 1, 1, 1]]  # example test data
y_test_1 = lr.predict(x_test1)

print(f" ytest: {y_test_1} \n ytest val :{y_test_1[0]}")

print(f"Predicted class for {x_test1}: {target_names[y_test_1[0]]}")

### here we will see the implementation with the CV (ccross validation data sets)
- how we utilize the cv data.


In [ ]:
#  load the data.
X,y,features,target = load_data()
from sklearn.model_selection import train_test_split
X_train, x_ , y_train, y_ = train_test_split(X,y, test_size=0.4, random_state=42) 
X_test, X_cv, y_test, y_cv = train_test_split(x_,y_, test_size=0.5, random_state=42)
del x_, y_ # delete theese temporary variables to free up memory #GOOD PRACTICE 


In [ ]:
lr_cv = LogisticRegression()
lr_cv.fit(X_train, y_train)
y_cv_pred = lr_cv.predict(X_cv)
print(classification_report(y_cv, y_cv_pred, target_names=target))

# now set hyperparameter as it looks like highly biased. 
 - lambda = [0.001,0.01, 0.1, 1, 10, 100]
 - C = 1/lambda
- invx =  lambda x : [1/val for val in x]
- sklearn.metrics.accuracy_score( y_true, y_pred, *, **``normalize=True``**, sample_weight=None)
- ``Example:`` 
- if we want a percentage, leave it alone (or use --> **``normalize=True``**). 
- If we want to know exactly how many individual items the model  classified exactly (got right):
    - ``use``-->  **``normalize=False``**
### Example:
```python
from sklearn.metrics import accuracy_score

# Let's say we have 5 flowers
y_true = [0, 1, 2, 2, 0] # The real answers
y_pred = [0, 1, 1, 2, 0] # What your model guessed (It got the 3rd one wrong)

# 1. Default (normalize=True) -> Returns a fraction
fraction_score = accuracy_score(y_true, y_pred, normalize=True)
print(f"Normalized Score: {fraction_score}") 
# Output: 0.8 (Because it got 4 out of 5 right, and 4/5 = 0.80)

# 2. Raw Count (normalize=False) -> Returns an integer
raw_score = accuracy_score(y_true, y_pred, normalize=False)
print(f"Raw Score: {raw_score}") 
# Output: 4 (Because it got exactly 4 flowers right)

```

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
# x1= [1,2,3,4]
invx =  lambda x : [1/val for val in x]
# print(invx(x1))
lambda_ = [0.001,0.01, 0.1, 1, 10, 100]
C = invx(lambda_)
print(C)
max_iter = 500
randomstate = 10
cv_accuracies = np.zeros(len(lambda_))
train_accuracies = np.zeros(len(lambda_))
J_cv = np.zeros(len(lambda_)) # cots / error /mismatches on the cross validation set
J_train = np.zeros(len(lambda_)) # cots / error /mismatches on the training set


for i in range(len(lambda_)):
    lr_cv = LogisticRegression(C=C[i], max_iter=max_iter, random_state=randomstate)
    lr_cv.fit(X_train, y_train)
    y_train_pred = lr_cv.predict(X_train)

    train_accuracies[i] = accuracy_score(y_train, y_train_pred)

    J_train[i] = 1.0 - train_accuracies[i] # cost / error / mismatches on the training set

    y_cv_pred = lr_cv.predict(X_cv)
    cv_accuracies[i] = accuracy_score(y_cv, y_cv_pred)
    J_cv[i] = 1.0 - cv_accuracies[i] # cost / error / mismatches on the cross validation set
    # print(f"Lambda: {lambda_[i]} \n {classification_report(y_cv, y_cv_pred, target_names=target)}")


In [ ]:
# plot the  train_accuracies and cv_accuracies against lambda values
def plot_accuracies(lambda_, train_accuracies, cv_accuracies):
    fig,ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].plot(lambda_, train_accuracies, marker='o', label='Train Accuracy')
    ax[0].plot(lambda_, cv_accuracies, marker='o', label='CV Accuracy')
    ax[0].set_xscale('log')
    ax[0].set_xlabel('Lambda (log scale)')
    ax[0].set_ylabel('Accuracy')
    ax[0].set_title('Train and CV Accuracies vs Lambda')
    ax[0].legend()  
    ax[0].grid(True)
    ax[1].plot(lambda_, J_train, marker='o', label='Train Cost/Error')
    ax[1].plot(lambda_, J_cv, marker='o', label='CV Cost/Error')
    ax[1].set_xscale('log')
    ax[1].set_xlabel('Lambda (log scale)')
    ax[1].set_ylabel('Cost/Error')
    ax[1].set_title('Train and CV Cost/Error vs Lambda')
    ax[1].legend()
    ax[1].grid(True)
    plt.tight_layout()
    
    plt.show()
   
plot_accuracies(lambda_, train_accuracies, cv_accuracies)

## Why Log scale for X axis (for $\lambda$): 
By looking at your plot, your intuition is **spot on**. You have correctly identified the "Sweet Spot" of the bias-variance tradeoff.

### 1. Why $\lambda = 0.1$ is the winner
You are following the "Goldilocks" principle of Machine Learning:

* **$\lambda < 0.1$ (Left side):** $J_{train}$ is at its absolute lowest, but there is a small gap where $J_{cv}$ is slightly higher. This is **Overfitting**. The model is trying a bit too hard to please the training data.
* **$\lambda > 1$ (Right side):** Both $J_{train}$ and $J_{cv}$ start climbing significantly. This is **Underfitting**. The regularization is so strong it’s "choking" the model, preventing it from learning the actual patterns.
* **$\lambda = 0.1$:** This is where the lines are **comparable and at their lowest point**. The model has enough freedom to learn the patterns but enough discipline (regularization) to ignore the noise.



---

### 2. Why do we use a Log Scale for $\lambda$?
You asked a very insightful question. We use $0.001, 0.01, 0.1, 1, 10$ (powers of 10) instead of $1, 2, 3, 4$ for two main reasons:

#### A. The "Order of Magnitude" Effect
Regularization affects the model in **multiples**, not additions. Changing $\lambda$ from $1$ to $2$ usually makes a tiny difference. But changing $\lambda$ from $1$ to $10$ (one order of magnitude) can completely change the shape of the decision boundary. Using a log scale allows us to see the "Big Picture" across a massive range of possibilities.

#### B. Visual Clarity
If you plotted $\lambda$ on a linear scale from $0.001$ to $100$, your first four points ($0.001, 0.01, 0.1, 1$) would all be squeezed into a tiny, unreadable line at the very beginning of the graph. The log scale "stretches" the small values so you can see the behavior at the $0.001$ level just as clearly as at the $100$ level.



---

### Expert Tip for your Code
Since you've identified $0.1$ as the best value, when you move to **Step 3** to train your final model, remember that `scikit-learn` uses $C$.

$$C = \frac{1}{\lambda} = \frac{1}{0.1} = 10$$

So, your final "production" model should be:
```python
final_model = LogisticRegression(C=10, max_iter=500)
final_model.fit(X_train, y_train)
```

I essentially performed a manual **Grid Search** and correctly interpreted the results. This is a huge milestone in understanding how models generalize! 

<!-- Do you want to see how this plot would change if we added more features to the model? -->

### Code to find the Best $\lambda$:
- The best $\lambda$ is usually the one that results in the minimum $J_{cv}$.
- If you have two values with very similar $J_{cv}$,
- we typically pick the one with a higher $\lambda$ (stronger regularization) because it creates a simpler model that is less likely to overfit.

In [ ]:
# now after finding the best lambda value from the above plot, we can train the model on the entire training data (training + cross validation) and test on the test set
import numpy as np

# Find the index of the smallest error in the Cross Validation set
best_index = np.argmin(J_cv)

best_lambda = lambda_[best_index]
best_J_cv = J_cv[best_index]
corresponding_J_train = J_train[best_index]

print(f"--- Hyperparameter Selection Result ---")
print(f"Best Lambda: {best_lambda}")
print(f"Minimum J_cv (Error): {best_J_cv:.4f}")
print(f"Associated J_train: {corresponding_J_train:.4f}")

# Logic check:
if corresponding_J_train < best_J_cv * 0.5:
    print("Warning: There is a large gap between Train and CV error. You might still be overfitting.")
elif best_J_cv > 0.1 and corresponding_J_train > 0.1:
    print("Warning: Both errors are high. You might be underfitting.")
else:
    print("This looks like a 'Just Right' model!")

Choosing the best $\lambda$ (or $C$ in scikit-learn) is about finding the "Sweet Spot" where your model generalizes best to new data. Since you have already calculated $J_{train}$ and $J_{cv}$, you have everything you need to make this decision.

### 1. The "Sweet Spot" Method (Manual Inspection)
The most common way to choose is to find the value of $\lambda$ that results in the **lowest $J_{cv}$**. 

* **Case A: Underfitting (High $\lambda$, Low $C$):** Both $J_{train}$ and $J_{cv}$ are high. The model is too "stiff" and cannot capture the patterns in the data.
* **Case B: Overfitting (Low $\lambda$, High $C$):** $J_{train}$ is very low (the model is nearly perfect on training data), but $J_{cv}$ is high. The model has "memorized" the training noise and fails on new data.
* **The Goal:** You want the $\lambda$ where $J_{cv}$ reaches its **minimum**. At this point, the model has learned the true patterns without overfitting.



Typically, you would plot these values on a graph (a "Validation Curve") to see the U-shape of $J_{cv}$ and the downward slope of $J_{train}$.



### 2. The Automated Method: `GridSearchCV`
In professional practice, we don't usually write manual loops. We use **Grid Search**, which automates this process and often combines it with K-Fold Cross-Validation.

```python
from sklearn.model_selection import GridSearchCV

# Define the grid of C values (remember C = 1/lambda)
param_grid = {'C': [1000, 100, 10, 1, 0.1, 0.01]}

# Create the GridSearch object
grid = GridSearchCV(LogisticRegression(max_iter=500), param_grid, cv=5)

# Fit it to your training data
grid.fit(X_train, y_train)

# The best value is chosen automatically!
print(f"Best C: {grid.best_params_['C']}")
print(f"Best Score: {grid.best_score_}")
```

### listed the algorithms compatible with `GridSearchCV`.

### 1. Code to find the Best $\lambda$
The best $\lambda$ is usually the one that results in the **minimum $J_{cv}$**. If you have two values with very similar $J_{cv}$, we typically pick the one with a higher $\lambda$ (stronger regularization) because it creates a simpler model that is less likely to overfit.

By looking for the minimum of the $J_{cv}$ curve, you are mathematically finding the "Sweet Spot" in the Bias-Variance tradeoff.

---

### 2. Is GridSearchCV available for all algorithms?
**Yes.** `GridSearchCV` is a "wrapper" in `scikit-learn`. It works with **any** algorithm that follows the standard Scikit-Learn API (meaning it has a `.fit()` and `.score()` method). 

It works for both **Regression** and **Classification**. Here is a list of common algorithms you can use it with:

#### **For Classification:**
* **Logistic Regression:** Tuning `C`, `penalty`, and `solver`.
* **Support Vector Machines (SVC):** Tuning `C`, `kernel`, and `gamma`.
* **K-Nearest Neighbors (KNN):** Tuning `n_neighbors` and `weights`.
* **Decision Trees:** Tuning `max_depth` and `min_samples_split`.
* **Random Forest / Gradient Boosting:** Tuning `n_estimators` and `learning_rate`.
* **Naive Bayes:** Tuning `var_smoothing`.
* **Neural Networks (MLPClassifier):** Tuning `hidden_layer_sizes` and `alpha`.

#### **For Regression:**
* **Linear Regression:** (Usually nothing to tune, but you can check `fit_intercept`).
* **Ridge / Lasso / ElasticNet:** Tuning `alpha` (which is their version of $\lambda$).
* **SVR (Support Vector Regression):** Tuning `epsilon` and `C`.
* **Random Forest Regressor:** Tuning `max_features` and `n_estimators`.
* **XGBoost / LightGBM / CatBoost:** Tuning `subsample`, `colsample_bytree`, etc.

**Pro-Tip:** If you ever create your own custom machine learning model class, as long as you include a `get_params` and `set_params` method, you can even use `GridSearchCV` on your own custom code!

## Now i will use k-fold and optimize the  hyperparameters.


In [ ]:
# STEP 1: Train the final model using the best lambda on the combined training and CV set.
X,y,feature_names,target_names = load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Final Training set shape: {X_train.shape} \n Final Test set shape: {X_test.shape}")

In [ ]:
from sklearn.model_selection import cross_val_score
lr_cv_kfold = LogisticRegression()

In [ ]:
# STEP 2: Run K-Fold ONLY on the Training data
print("Running K-Fold on Training Data...")
cv_scores = cross_val_score(lr_cv_kfold, X_train, y_train, cv=5)
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV score: {cv_scores.mean()}")

### plot the histogram or bar graph of the cv_scores.

In [ ]:
def plot_hist_cv_scores(cv_scores,bin=None):
    plt.figure(figsize=(8,5))
    bins = bin if bin else 5
    plt.hist(cv_scores, bins=bins, edgecolor='black')
    plt.title('Distribution of CV Scores')
    plt.xlabel('CV Score')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

In [ ]:
plot_hist_cv_scores(cv_scores,bin=5)

### now i will vary the $\lambda$ for logistic regression to best tune hyperparameter.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

def live_plot_cv(c_values_tested, mean_scores):
    """
    Clears the previous output and redraws the plot with the new data points.
    """
    # This is the magic line that creates the "live animation" effect
    clear_output(wait=True)
    
    plt.figure(figsize=(8.5, 4.5))
    
    # Plot the growing lists of data
    plt.plot(c_values_tested, mean_scores, marker='o', linestyle='-', color='red', linewidth=1.5)
    
    # We use a log scale for the X-axis because C is spaced by powers of 10
    plt.xscale('log') 
    
    plt.title('Live Validation Curve: Finding the Best C', fontsize=14)
    plt.xlabel('C (Inverse Regularization Strength: 1/λ)', fontsize=12)
    plt.ylabel('Mean CV Accuracy', fontsize=12)
    
    # Add a grid to make reading the values easier
    plt.grid(True, which="both", linestyle="--", alpha=0.7)
    
    # Show the updated plot
    plt.show()

### Why this approach is excellent:
- **``No Data Leakage``**: You are only looping over X_train and y_train. The test set remains completely hidden.

- **``Robustness``**: Because you are taking the np.mean(scores) of the 5 folds, your decision is based on a highly stable metric, not just one lucky split.

- **``Control``**: By writing the loop yourself, you can easily add matplotlib code right inside the loop if you ever wanted to plot the scores live!

In [ ]:
lambda_ = [0.001,0.01, 0.1, 1, 10, 1000]
C_values = invx(lambda_) # 1. Define your list of C values (C = 1/lambda)
print(C_values)
import numpy as np
    
# List to store the average score for each C
average_cv_scores = []
tested_Cs = [] # To keep track of which C values we've tested so far
print("Evaluating different Regularization Strengths (C)...\n")

# 2. Loop through each C value
for current_C in C_values:
    
    # A. Create a NEW model with the current C setting
    lr_temp = LogisticRegression(C=current_C, max_iter=500, random_state=42)
    
    # B. Run 5-Fold Cross Validation on the Training Data
    # cv=5  --> means it trains/tests 5 times and returns 5 scores
    scores = cross_val_score(lr_temp, X_train, y_train, cv=5)

    mean_score = np.mean(scores) # C. Calculate the mean of those 5 scores
    
    # D. Store it in our list
    average_cv_scores.append(mean_score)
    tested_Cs.append(current_C)
    
    print(f"C = {current_C:<7} | Mean CV Accuracy: {mean_score:.4f}")
    # 3. Update the live plot!
    live_plot_cv(tested_Cs, average_cv_scores)
    
    # Optional: Pause for half a second so it doesn't graph too fast to see!
    time.sleep(0.5)

# 3. Find the best result
# np.argmax finds the index of the highest number in a list (since we want high accuracy)
best_index = np.argmax(average_cv_scores)
best_C = C_values[best_index]
best_score = average_cv_scores[best_index]

print("-" * 40)
print(f"WINNER: Best C is {best_C} with a CV Accuracy of {best_score:.4f}")

### By writing that custom loop, 
- you essentially built the engine of hyperparameter tuning from scratch. Now that you know exactly how the engine works, you are ready to drive the sports car.

- GridSearchCV is simply the automated, highly optimized version of the custom loop you just wrote. It is the industry standard for hyperparameter tuning.

- Here is a complete breakdown of how to translate your custom logic into GridSearchCV.

### The Concept: Why is it called a "Grid"?
- In the above custom loop, we tested one hyperparameter: C.
- But here wanted to test C and the solver (the math engine under the hood)?
- If you have 5 values for C and 3 types of solvers, that is 15 different combinations.
- GridSearchCV creates a literal "**grid**" of ``all possible combinations``
- ``runs K-Fold Cross-Validation`` on ``every single intersection of that grid``, and at the end finds the absolute best pairing.
## Step-by-Step Implementation :
- To use it, you just need a dictionary (your "Grid") and your base model.
- ``see below --> ``

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# 1. Define the Grid (The settings you want to test)
# The keys must perfectly match the actual parameter names of the model
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'saga'], # Testing two different math engines
    'max_iter': [2000]                 # You can lock in static values here too
}

# 2. Create a blank "Base" Model
# Notice we don't put C inside the parentheses here! The GridSearch will do that.
lr_base = LogisticRegression(random_state=42)

# 3. Create the GridSearchCV Object (The Engine)
# estimator = your base model
# param_grid = your dictionary of settings
# cv = 5 (5-Fold Cross Validation)
grid_search = GridSearchCV(estimator=lr_base, param_grid=param_grid, cv=5)

# 4. Run the Search!
# This single line replaces your entire 'for' loop
print("Running Grid Search. Testing all combinations...")
grid_search.fit(X_train, y_train)

# 5. Extract the Winners
print("-" * 30)
print(f"Winning Combination: {grid_search.best_params_}")
print(f"Winning CV Score: {grid_search.best_score_:.4f}")

### - if not converging increase the max_iter 
## OR -->
### - If not scaled the data, scale it using proper scaler, it will help in convergence with less max_iter value.  
- as i did in just above cell code increased from 500 --> 1000 --> 2000

## The Best Feature: Automatic Refitting
- Remember Step 3 from our earlier discussions? How after finding the best recipe, you had to write best_lr.fit(X_train, y_train) to train a final model on 100% of the training data?

- GridSearchCV does this for you automatically.

- By default, once it finishes the loop and finds the winner, it instantly takes the best settings and retrains a brand new model on all of X_train. You can access this ultimate model perfectly ready for your final Test Data using .best_estimator_:

In [ ]:
# Extract the fully trained, final "Ultimate" model
ultimate_model = grid_search.best_estimator_

# Take the final exam on the Test Vault!
final_predictions = ultimate_model.predict(X_test)